# Inverse 2-D Heat-Transfer PINN (thin notebook)

**Author:** Ivan Lisovskyi

**Goal:** reconstruct the temperature field in a 2-D plate and estimate the unknown thermal diffusivity $\alpha$ from sparse COMSOL sensor data.

This notebook only calls the package in `src/pinn/`: the model, physics, training loop and figures were moved there from the original single-file notebook, [`final_notebook.ipynb`](final_notebook.ipynb), which is kept unchanged (it also holds the explanations and the academic references). The same run is available from the command line: `python -m pinn train`.

Setup (from the project folder): `python -m pip install -r requirements.txt` and `python -m pip install -e . --no-deps`.

## 1. Settings

All constants live in `pinn.config.Config`; its defaults are the original notebook's values (2,000 epochs with fixed diffusivity, then 6,000 with trainable diffusivity). **The training cell below is not a short demo run.** For a quick look, lower `stage1` and `stage2`, for example `Config(stage1=100, stage2=300, log_every=50)`; the results will then not match the saved study. The reference value `alpha_true = 1e-5 m²/s` is only used to report errors, not in the training loss.

In [ ]:
import matplotlib.pyplot as plt

from pinn.config import Config, resolve_device
from pinn.data import PROJECT_DIR, load_comsol_sensor_data
from pinn.plots import (plot_centerline, plot_contours, plot_sensor_fit,
                        plot_sensor_layout, plot_training_dashboard, save_figures)
from pinn.results import (alpha_sensitivity_table, architecture_summary,
                          load_saved_results, markdown_table)
from pinn.train import alpha_error_pct, save_run, train

cfg = Config()                              # defaults = the original notebook's constants
OUT_DIR = PROJECT_DIR / "outputs_submission"  # git-ignored; never overwrites results/
print("device:", resolve_device(cfg.device))

## 2. COMSOL sensor data

`sensors_T.csv.xz` (or an unpacked `sensors_T.csv`) is found automatically. Time is in seconds, temperature in kelvin, coordinates in metres.

In [ ]:
data = load_comsol_sensor_data()
t_np, x_np, y_np, T_np = data
print(f"timesteps={T_np.shape[0]}, sensors={T_np.shape[1]}, max rise={T_np.max() - cfg.t0:.2f} K")
plot_sensor_layout(x_np, y_np, cfg)
plt.show()

## 3. Train

Stage 1 fits the temperature field with $\alpha$ frozen; stage 2 trains the network and $\alpha$ together. 20 % of the sensors are held out for validation.

In [ ]:
model, hist, data = train(cfg)
save_run(model, hist, cfg, OUT_DIR)   # history.csv, metrics.txt, checkpoint.pt

## 4. Figures

Post-processing only; nothing here changes the trained model.

In [ ]:
t_np, x_np, y_np, T_np, train_s, val_s = data
plot_sensor_layout(x_np, y_np, cfg, train_s, val_s)
plot_training_dashboard(hist, cfg)
plot_sensor_fit(model, data, cfg)
plot_centerline(model, data, cfg)
plot_contours(model, cfg)
plt.show()

In [ ]:
paths = save_figures(model, hist, data, cfg, OUT_DIR)   # same four PNGs as `python -m pinn train`
print("saved:", *paths, sep="\n  ")

## 5. Saved study results

Read from the files committed in the repository (`results/selected-run/metrics.txt`, `outputs_alpha_sensitivity/`, `outputs_comparison/`); nothing is trained here. These are the saved project results, not the run above.

In [ ]:
saved = load_saved_results()
sel = saved["selected_run"]
print(f"selected run: alpha = {sel['alpha']:.6e} m^2/s, error {sel['alpha_error_pct']:.3f} %, "
      f"held-out RMSE {sel['val_rmse_K']:.6f} K")
alpha_sensitivity_table(saved["alpha_sensitivity"])

In [ ]:
architecture_summary(saved["comparison"]).round(3)